## PKR DEPRECIATION & CPI PASS-THROUGH ANALYSIS

In [11]:
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine, text

In [12]:
df = pd.read_csv("pakistan_economic_indicators_2000_2025.csv")

print("Shape:", df.shape)
print("Nulls:\n", df.isnull().sum().sum(), "total null values")
print("Year range:", df['year'].min(), "-", df['year'].max())

Shape: (26, 32)
Nulls:
 0 total null values
Year range: 2000 - 2025


In [13]:
# Year-over-year PKR depreciation (positive = weaker PKR)
df['yoy_depreciation_pct'] = df['pkr_per_usd'].pct_change() * 100

# Year-over-year CPI change
df['cpi_change_pct'] = df['inflation_cpi_pct'].pct_change() * 100


In [14]:
# Pass-through coefficient: how much of depreciation fed into CPI
# Formula: CPI level / YoY depreciation pct (standard ERPT coefficient)
df['pass_through_coefficient'] = df['inflation_cpi_pct'] / df['yoy_depreciation_pct']

# Clean up inf and div-by-zero (years with zero depreciation)
df['pass_through_coefficient'] = df['pass_through_coefficient'].replace(
    [float('inf'), float('-inf')], None
)

In [15]:
# Round engineered columns
df['yoy_depreciation_pct']    = df['yoy_depreciation_pct'].round(2)
df['cpi_change_pct']          = df['cpi_change_pct'].round(2)
df['pass_through_coefficient'] = df['pass_through_coefficient'].round(4)


In [16]:
# LABEL DEPRECIATION EPISODES

def label_episode(year):
    if year in [2018, 2019]:
        return 'Episode_2018_TwinDeficits'
    elif year in [2022, 2023]:
        return 'Episode_2022_PKRCrash'
    else:
        return 'No_Episode'

df['episode'] = df['year'].apply(label_episode)

print("\nEngineered columns preview:")
print(df[['year', 'pkr_per_usd', 'yoy_depreciation_pct',
          'inflation_cpi_pct',
          'pass_through_coefficient', 'episode']].to_string())



Engineered columns preview:
    year  pkr_per_usd  yoy_depreciation_pct  inflation_cpi_pct  pass_through_coefficient                    episode
0   2000         51.8                   NaN                4.4                       NaN                 No_Episode
1   2001         61.4                 18.53                3.1                    0.1673                 No_Episode
2   2002         59.7                 -2.77                3.5                   -1.2641                 No_Episode
3   2003         57.8                 -3.18                3.1                   -0.9741                 No_Episode
4   2004         57.6                 -0.35                7.4                  -21.3860                 No_Episode
5   2005         59.5                  3.30                9.3                    2.8194                 No_Episode
6   2006         60.4                  1.51                7.9                    5.2228                 No_Episode
7   2007         60.9                  0.83

In [17]:
# EXPORT CLEAN CSV (backup)

df.to_csv("pkr_clean.csv", index=False)
print("\npkr_clean.csv saved successfully.")



pkr_clean.csv saved successfully.


In [ ]:
# PUSH TO MYSQL

DB_USER     = "root"
DB_PASSWORD = "password"      
DB_HOST     = "localhost"
DB_PORT     = "3306"
DB_NAME     = "pkr_analysis"        

# First create the database if it doesn't exist
engine_no_db = create_engine(
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/"
)
with engine_no_db.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))
    print(f"\nDatabase '{DB_NAME}' ready.")

# Now connect to the database
engine = create_engine(
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Push main table
df.to_sql(
    name="economic_indicators",
    con=engine,
    if_exists="replace",
    index=False
)
print("Table 'economic_indicators' loaded:", len(df), "rows.")



Database 'pkr_analysis' ready.
Table 'economic_indicators' loaded: 26 rows.


In [19]:
# CREATE EPISODE SUMMARY TABLE
episode_summary = df[df['episode'] != 'No_Episode'].groupby('episode').agg(
    start_year        = ('year', 'min'),
    end_year          = ('year', 'max'),
    pkr_start         = ('pkr_per_usd', 'first'),
    pkr_end           = ('pkr_per_usd', 'last'),
    avg_depreciation  = ('yoy_depreciation_pct', 'mean'),
    avg_cpi           = ('inflation_cpi_pct', 'mean'),
    avg_pass_through  = ('pass_through_coefficient', 'mean'),
    avg_policy_rate   = ('policy_rate_pct', 'mean'),
    avg_forex_reserves= ('forex_reserves_usd_bn', 'mean')
).reset_index()

episode_summary['total_depreciation_pct'] = (
    (episode_summary['pkr_end'] - episode_summary['pkr_start'])
    / episode_summary['pkr_start'] * 100
).round(2)

episode_summary = episode_summary.round(4)

episode_summary.to_sql(
    name="depreciation_episodes",
    con=engine,
    if_exists="replace",
    index=False
)
print("Table 'depreciation_episodes' loaded:", len(episode_summary), "rows.")




Table 'depreciation_episodes' loaded: 2 rows.
